### 基于内存的短期记忆

In [3]:
## 基于内存的记忆
import os

from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from langchain_core.messages import AIMessage, SystemMessage, ToolMessage, HumanMessage
from langgraph.checkpoint.memory import InMemorySaver

load_dotenv(override=True)

model = init_chat_model(
    api_base=os.getenv("DEEPSEEK_API_BASE"),
    api_key=os.getenv("DEEPSEEK_API_KEY"),
    model="deepseek-flash",
    model_provider="deepseek",
    model_kwargs={"reasoning_effort": "none"},
)

In [4]:
checkpoint = InMemorySaver()

agent = create_agent(
    model=model,
    name="deepseek_agent",
    checkpointer=checkpoint
)

config = {
    "configurable":{
        "thread_id":"1"
    }
}
messages = [
    SystemMessage("你是一个小助手，用杀生鱼丸的口吻和我说话")
]
while False:
    user_input = input("请输入：")
    messages.append(HumanMessage(user_input))
    response = agent.invoke({"messages": messages}, config=config)
    print(response["messages"][-1].content)



### 基于持久化的短期记忆

`InMemorySaver` 把记忆存在进程内存里，**一旦进程（Kernel）结束，记忆就全部丢失**。持久化短期记忆就是换一个把检查点写入外部存储的 checkpointer，这样只要 `thread_id` 不变，即使重启程序也能恢复完整对话。

| Checkpointer | 存储介质 | 适用场景 |
| --- | --- | --- |
| `InMemorySaver` | 进程内存 | 测试/演示，进程结束即丢失 |
| `SqliteSaver` | SQLite 文件（同步） | 本地单机持久化 |
| `AsyncSqliteSaver` | SQLite 文件（异步） | 异步框架 + 本地持久化 |
| `PostgresSaver` | PostgreSQL（同步） | 生产环境、多实例共享 |
| `AsyncPostgresSaver` | PostgreSQL（异步） | 异步生产环境 |

> 本章使用 SQLite：无需额外服务，直接落地成 `.db` 文件，最适合本地演示。

依赖：`langgraph-checkpoint-sqlite`（已通过 `uv add langgraph-checkpoint-sqlite` 加入项目）。

In [5]:
import sqlite3
from pathlib import Path

from langgraph.checkpoint.sqlite import SqliteSaver

# 记忆文件放在项目 temp/ 目录下（已在 .gitignore 中忽略）
DB_DIR = Path.cwd() / "temp"
DB_DIR.mkdir(exist_ok=True)
DB_PATH = DB_DIR / "short_memory.db"


def make_persistent_agent():
    """创建一个使用 SQLite 持久化记忆的 agent。

    每次调用都新建连接，用来模拟「重启进程」。
    check_same_thread=False：允许连接跨线程使用（LangGraph 内部会用到）。
    """
    conn = sqlite3.connect(DB_PATH, check_same_thread=False)
    saver = SqliteSaver(conn)
    agent = create_agent(model=model, checkpointer=saver)
    return agent, saver, conn


#### 步骤 1：在 thread 中写入记忆

有了 checkpointer 后，**只需要传入本轮的新消息**，历史会自动从检查点恢复并按 `thread_id` 关联。

In [6]:
# 清掉旧的演示库，保证可重复运行
if DB_PATH.exists():
    DB_PATH.unlink()

agent, saver, conn = make_persistent_agent()

thread_id = "user-persist-1"
config = {"configurable": {"thread_id": thread_id}}

agent.invoke({"messages": [{"role": "user", "content": "你好，我叫张三。"}]}, config=config)
result = agent.invoke({"messages": [{"role": "user", "content": "我最爱吃兰州拉面。"}]}, config=config)
print("第一次回复：", result["messages"][-1].content)

# 关闭连接，等价于「进程结束」。数据此时已写入磁盘文件
conn.close()
print("已关闭连接，记忆已持久化到：", DB_PATH)


第一次回复： 兰州拉面确实很美味！🍜 汤鲜面劲，配上几片牛肉和香菜，想想就让人流口水。你最喜欢哪家的？或者你喜欢毛细、二细还是宽面？
已关闭连接，记忆已持久化到： /Users/lijixu/PycharmProjects/langchain_demo/charpter10-memery/temp/short_memory.db


#### 步骤 2：模拟重启后恢复记忆

新建一个全新的连接和 agent 对象（相当于重启了程序 / Kernel），只要 `thread_id` 相同，就能读到之前的对话。这是「持久化」与「内存记忆」最本质的区别。

In [7]:
agent_restart, saver_restart, conn_restart = make_persistent_agent()

result = agent_restart.invoke(
    {"messages": [{"role": "user", "content": "你还记得我叫什么吗？我爱吃什么？"}]},
    config=config,
)
print("重启后回复：", result["messages"][-1].content)


重启后回复： 当然记得！你叫**张三**，最爱吃**兰州拉面**。🍜

有什么需要我帮忙的吗，张三？


#### 步骤 3：查看与管理检查点

In [8]:
# 查看当前线程的最新状态（检查点）
snapshot = agent_restart.get_state(config)
print("当前线程消息数：", len(snapshot.values["messages"]))
print("下一步将执行的节点：", snapshot.next)
print("latest checkpoint_id：", snapshot.config["configurable"].get("checkpoint_id"))

# 查看该线程的完整检查点历史
history = list(agent_restart.get_state_history(config))
print("历史检查点数量：", len(history))


当前线程消息数： 6
下一步将执行的节点： ()
latest checkpoint_id： 1f1b743d-09b1-6f5e-8007-72306f05b732
历史检查点数量： 9


#### 步骤 4：thread 之间互相隔离

记忆按 `thread_id` 隔离：换个 `thread_id` 就是一段全新的对话。

In [9]:
other_config = {"configurable": {"thread_id": "user-persist-2"}}
result = agent_restart.invoke(
    {"messages": [{"role": "user", "content": "我叫什么名字？"}]},
    config=other_config,
)
print("新线程回复：", result["messages"][-1].content)


新线程回复： 你好！很抱歉，我在这次对话中还没有得知你的名字。如果你愿意告诉我，我会很高兴记住它！😊

你可以直接告诉我你想让我怎么称呼你，这样以后交流起来会更亲切。有什么我可以帮你的吗？


#### 步骤 5：删除某个线程的记忆

In [10]:
saver_restart.delete_thread("user-persist-2")
print("已删除 user-persist-2 的记忆")

# 验证：删除后再问，它已经不记得了
result = agent_restart.invoke(
    {"messages": [{"role": "user", "content": "我叫什么名字？"}]},
    config=other_config,
)
print("删除后回复：", result["messages"][-1].content[:40])


已删除 user-persist-2 的记忆
删除后回复： 你好！😊 从我们的对话记录来看，你还没有告诉我你的名字呢。如果你愿意的话，可以现


#### 异步版本：AsyncSqliteSaver

如果用异步方式（`ainvoke` / `astream`），请改用 `AsyncSqliteSaver`，它基于 `aiosqlite`，用法是异步上下文管理器。

`AsyncSqliteSaver.from_conn_string(...)` 返回的 saver 生命周期与 `async with` 绑定，退出代码块后连接会关闭。

In [11]:
from langgraph.checkpoint.sqlite.aio import AsyncSqliteSaver

DB_AIO_PATH = DB_DIR / "short_memory_aio.db"


async def run_async_persist_demo():
    if DB_AIO_PATH.exists():
        DB_AIO_PATH.unlink()

    async with AsyncSqliteSaver.from_conn_string(str(DB_AIO_PATH)) as saver:
        async_agent = create_agent(model=model, checkpointer=saver)
        cfg = {"configurable": {"thread_id": "user-aio-1"}}

        await async_agent.ainvoke(
            {"messages": [{"role": "user", "content": "记住：我叫李四。"}]}, config=cfg
        )
        result = await async_agent.ainvoke(
            {"messages": [{"role": "user", "content": "我叫什么名字？"}]}, config=cfg
        )
        print("异步持久化回复：", result["messages"][-1].content)


await run_async_persist_demo()


异步持久化回复： 你叫李四。


### PostgreSQL 持久化

In [ ]:
import os

from langchain.agents import create_agent
from langchain_core.messages import HumanMessage
from langgraph.checkpoint.postgres import PostgresSaver
from dotenv import load_dotenv
load_dotenv(override=True)


with PostgresSaver.from_conn_string(os.getenv("DATABASE_URL")) as checkpointer:
    # 初始化数据库
    checkpointer.setup()

    agent = create_agent(model="deepseek:deepseek-flash", checkpointer=checkpointer)

    config = {
        "configurable":{
            "thread_id":"1"
        }
    }

    response = agent.invoke({
            "messages":[HumanMessage("我叫小许")]
        }, config=config)

    response1 = agent.invoke({
            "messages":[HumanMessage("我叫什么")]
        }, config=config)
    print(response1["messages"][-1].pretty_print())



#### 什么时候用哪种？

| 需求 | 推荐 |
| --- | --- |
| 本地开发、单机演示 | `SqliteSaver` |
| 异步应用、本地 | `AsyncSqliteSaver` |
| 生产、多实例共享 | `PostgresSaver` / `AsyncPostgresSaver` |

**要点**

1. 持久化 = 把 checkpointer 从 `InMemorySaver` 换成外部存储实现；
2. `thread_id` 是记忆的隔离键，也是跨进程恢复记忆的钥匙；
3. 有 checkpointer 时只传新消息，历史由检查点自动恢复；
4. 同步库配同步调用（`invoke`），异步库配异步调用（`ainvoke`）；
5. `get_state` / `get_state_history` 可检查与管理记忆，`delete_thread` 可清除记忆。